In [0]:
dbutils.widgets.removeAll()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("golden_schema", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
silver_schema = dbutils.widgets.get("silver_schema")
golden_schema = dbutils.widgets.get("golden_schema")

In [0]:
silver_df = spark.table(f"{catalogo}.{silver_schema}.olist_orders_enriched")

In [0]:
golden_base_df = silver_df.withColumn(
    "year",
    year(col("order_purchase_date"))
).withColumn(
    "month",
    month(col("order_purchase_date"))
).withColumn(
    "year_month",
    date_format(col("order_purchase_date"), "yyyy-MM")
)

In [0]:
golden_summary_df = golden_base_df.groupBy(
    col("year"),
    col("month"),
    col("year_month"),
    col("customer_state"),
    col("customer_region"),
    col("product_category_name_english"),
    col("payment_type"),
    col("delivery_status")
).agg(
    countDistinct("order_id").alias("total_orders"),
    countDistinct("customer_unique_id").alias("total_customers"),
    count("order_item_id").alias("total_items_sold"),
    sum("price").alias("total_revenue"),
    sum("freight_value").alias("total_freight"),
    sum("payment_value").alias("total_payment_value"),
    avg("price").alias("average_item_price"),
    avg("payment_value").alias("average_payment_value"),
    avg("delivery_days").alias("average_delivery_days"),
    sum(when(col("is_late_delivery") == True, 1).otherwise(0)).alias("late_orders")
)

In [0]:
golden_final_df = golden_summary_df.withColumn(
    "late_delivery_rate",
    when(col("total_orders") > 0, col("late_orders") / col("total_orders")).otherwise(0)
).select(
    col("year"),
    col("month"),
    col("year_month"),
    col("customer_state"),
    col("customer_region"),
    col("product_category_name_english"),
    col("payment_type"),
    col("delivery_status"),
    col("total_orders"),
    col("total_customers"),
    col("total_items_sold"),
    col("total_revenue"),
    col("total_freight"),
    col("total_payment_value"),
    col("average_item_price"),
    col("average_payment_value"),
    col("average_delivery_days"),
    col("late_orders"),
    col("late_delivery_rate")
)

In [0]:
golden_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{golden_schema}.olist_sales_summary")

In [0]:
display(golden_final_df.limit(10))